# Vietnamese Text Classification with Mixture of Experts experiment

This notebook prepares a Colab runtime, but training and evaluation are run manually from shell commands. That keeps the workflow explicit: edit a command, run it, and reuse the resulting `run_id` when evaluating.

Use the command cells below to:
- mount Drive and install the project
- authenticate Hugging Face and W&B through Colab's built-in Secrets panel
- run a smoke test or a full experiment
- evaluate the saved checkpoint
- compare saved metrics in the final summary cell

Create the `HF_TOKEN` and `WANDB_API_KEY` secrets in Colab before running the authentication cell. The credentials are stored by the respective login tools and are not written to the notebook, an `.env` file, or the repository.

The repository code remains the source of truth. Each run saves its resolved configuration, checkpoint, metrics, and predictions.

## 1. Mount Google Drive

The YAML checkpoint path points to `/content/drive/MyDrive/ViHSD-MoE/checkpoints`. Drive must be mounted before training so `.safetensors` files persist after the Colab runtime ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the GitHub repository and install dependencies

This notebook treats GitHub as the source of truth. Each runtime clones the latest `main` branch into `/content/moe-vihsd`, installs dependencies from that clone, and runs the scripts there.

In [ ]:
PROJECT_DIR = '/content/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'

!rm -rf $PROJECT_DIR
!git clone --depth 1 --branch main $REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR
%pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Create a Colab Secret named HF_TOKEN before continuing.')
os.environ['HF_TOKEN'] = hf_token

wandb_api_key = userdata.get('WANDB_API_KEY')
if not wandb_api_key:
    raise RuntimeError('Create a Colab Secret named WANDB_API_KEY before continuing.')
os.environ['WANDB_API_KEY'] = wandb_api_key

import wandb
wandb.login(verify=True)
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/checkpoints'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/results'
print('Hugging Face and W&B authentication configured from Colab Secrets.')

## 3. Run training and evaluation manually

The repository uses one shared training pipeline with four architecture families:

- `dense_transformer`: dense Hugging Face encoder and classifier.
- `transformer_moe`: selected internal Transformer FFNs replaced by token-level MoE.
- `dynamic_moe`: dense encoder with sequence-level dynamic-threshold routing.
- `classifier_moe`: dense encoder with a pooled-representation MoE classification head.

Use canonical architecture names for new runs. The older `dense_phobert` and `phobert_moe` names remain compatibility aliases. Every run saves its resolved configuration, checkpoint, metrics, and predictions.

### Smoke test

```bash
python train.py --config configs/vihsd.yaml --smoke-test --run-id smoke-check
python evaluate.py --config configs/vihsd.yaml --run-id smoke-check
```

### Dense Transformer baseline

```bash
python train.py \
  --config configs/vihsd.yaml \
  --no-smoke-test \
  --run-id dense-transformer-integrated \
  --set model.architecture=dense_transformer \
  --set model.pooling=mean \
  --set training.loss_type=cross_entropy
python evaluate.py --config configs/vihsd.yaml --run-id dense-transformer-integrated
```

### Transformer-level MoE

```bash
python train.py \
  --config configs/vihsd.yaml \
  --no-smoke-test \
  --run-id transformer-moe-integrated \
  --set model.architecture=transformer_moe \
  --set training.loss_type=cross_entropy
python evaluate.py --config configs/vihsd.yaml --run-id transformer-moe-integrated
```

Use a unique `--run-id` for every experiment. Inspect `checkpoints/<run-id>/resolved_config.yaml` first when debugging.

## 4. Training command

Edit the command below and run the cell manually. The command prints the run ID and writes the checkpoint and metrics for that run.

For repeated experiments, change `--run-id` and any `--set` values before running the cell. Do not reuse a run ID unless you intentionally want to replace or inspect that run.

In [ ]:
# Edit the run_id before executing this cell.
# Dense Transformer baseline:
!python train.py --config configs/vihsd.yaml --no-smoke-test --run-id dense-transformer-integrated --set model.architecture=dense_transformer --set model.pooling=mean --set training.loss_type=cross_entropy

# Transformer-level MoE:
# !python train.py --config configs/vihsd.yaml --no-smoke-test --run-id transformer-moe-integrated --set model.architecture=transformer_moe --set training.loss_type=cross_entropy

## 5. Evaluation command

Run evaluation after the training command finishes. Use the same `run_id` so evaluation loads that run's `resolved_config.yaml` and best checkpoint. Evaluation writes classification metrics, predictions, confusion-matrix counts, and class-distribution diagnostics under `results/<run-id>/`.

In [ ]:
# Use the run_id from the training command above.
!python evaluate.py --config configs/vihsd.yaml --run-id transformer-moe-integrated

# Or evaluate a checkpoint directly:
# !python evaluate.py --config configs/vihsd.yaml --checkpoint checkpoints/transformer-moe-integrated/transformer_moe_best.safetensors

## 6. Command-line and configuration reference

Run commands from the repository root (`/content/moe-vihsd` in Colab). Configuration roles and inheritance rules are documented in `configs/README.md`.

### Command-line arguments

| Command | Argument | Default | What it does |
| --- | --- | --- | --- |
| `train.py` | `--config PATH` | `configs/vihsd.yaml` | Loads the YAML configuration and optional `base_config`. |
| `train.py` | `--set SECTION.KEY=VALUE` | none | Overrides an existing YAML value; repeat it for multiple overrides. |
| `train.py` | `--smoke-test` / `--no-smoke-test` | config value (`false`) | Selects the short validation profile or full training. |
| `train.py` | `--run-id ID` | timestamped ID | Names the checkpoint and results directory. |
| `evaluate.py` | `--config PATH` | `configs/vihsd.yaml` | Supplies the configuration used to rebuild the model and data pipeline. |
| `evaluate.py` | `--run-id ID` | latest run | Evaluates a saved run using its resolved configuration. |
| `evaluate.py` | `--checkpoint PATH` | none | Evaluates one explicit `.safetensors` checkpoint. |

### Configuration arguments

These defaults come from `configs/vihsd.yaml`. Use `--set section.key=value` to override a value for one run.

| Key | Default | What it does |
| --- | --- | --- |
| `seed` | `42` | Seeds Python, NumPy, and PyTorch. |
| `dataset.name` | `data/vihsd_segmented` | Dataset path or Hugging Face dataset ID. |
| `dataset.max_length` | `128` | Maximum token sequence length. |
| `model.architecture` | `transformer_moe` | Selects `dense_transformer`, `transformer_moe`, `dynamic_moe`, or `classifier_moe`. |
| `model.pretrained_model_name` | `vinai/phobert-base` | Encoder and tokenizer model ID or local path. Change this for XLM-R or mBERT. |
| `model.moe_layers` | `[8, 9, 10, 11]` | Zero-based layers whose FFNs are replaced by Transformer MoE. |
| `model.num_experts` | `4` | Number of routed experts. |
| `model.top_k` | `1` | Experts selected for each token or example. |
| `model.routing_method` | `top_k` | Transformer-MoE routing method: `top_k` or `dynamic_threshold`. |
| `model.upcycle` | `true` | Initializes GELU experts from pretrained FFN weights. |
| `model.shared_expert` | `false` | Adds an always-active expert. |
| `model.expert_type` | `gelu` | Transformer expert type: `gelu`, `geglu`, or `swiglu`. |
| `model.freeze_backbone` | `false` | Freezes the complete encoder for dense or classifier-level MoE models. |
| `model.pooling` | `cls` | Uses the first token or mean of non-padding tokens. |
| `training.epochs` | `5` | Number of full epochs. |
| `training.batch_size` | `16` | Examples per optimization step. |
| `training.learning_rate` | `0.00002` | AdamW learning rate. |
| `training.weight_decay` | `0.01` | AdamW weight decay. |
| `training.loss_type` | `cross_entropy` | Task loss type. |
| `routing.load_balance_loss_factor` | `0.01` | Weight of the auxiliary MoE balancing loss. |
| `logging.use_wandb` | `true` | Enables W&B logging. |

The model name controls both encoder and tokenizer, so XLM-R requires only:

```bash
python train.py --config configs/vihsd.yaml --no-smoke-test \
  --run-id xlmr-transformer-moe-s42 \
  --set model.architecture=transformer_moe \
  --set model.pretrained_model_name=xlm-roberta-base
```

For dense mBERT, use `model.architecture=dense_transformer`. Use the dedicated overlays under `configs/architectures/`, `configs/stages/`, and `configs/diagnostics/` when they match the experiment purpose.
